# 07 — Ensemble (voting + stacking)

Combina os 4 modelos base do artigo (decisao H):

| Base model | Notebook |
|---|---|
| `tfidf_logreg` | 02 |
| `tfidf_linearsvc` | 03 |
| `tfidf_multinomialnb` | 04 |
| `bert_bertimbau_base` | 05 |

**Estrategias** (binario):
- `majority_vote` — pelo menos 3 dos 4 votam 1
- `weighted_vote` — score medio ponderado por F1, threshold 0.5
- `stacking` — meta-classificador (LogReg) treinado nos scores da val, aplicado em test (so para `test_set`)

**Estrategias** (multiclasse):
- `majority` — classe mais votada (mode); empate vai para a classe predita pelo modelo individual com maior macro_F1

Cada estrategia emite um `result_card.json` com `model_id` no formato `ensemble_{strategy}` e custo = soma dos custos dos base models.

**Pre-requisito**: notebooks 02, 03, 04, 05 executados — os result cards e predictions devem estar em `RUNS_BASE/`.

## 0. Bootstrap

In [ ]:
import os, sys, zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("Ambiente:", "Google Colab" if IN_COLAB else "Local")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = Path("/content/drive/MyDrive/economy-classifier")
    REPO_DIR = Path("/content/economy-classifier")
    if not REPO_DIR.exists():
        os.system(f"git clone --branch main https://github.com/almeidadm/economy-classifier.git {REPO_DIR}")
    os.system(f"pip install -e {REPO_DIR} -q")
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    if not (SPLITS_DIR / "train.parquet").exists():
        with zipfile.ZipFile(DRIVE_BASE / "colab_splits.zip") as zf:
            zf.extractall(REPO_DIR / "artifacts")
    RUNS_BASE = DRIVE_BASE / "runs"
    HARDWARE = "Colab-CPU"
else:
    REPO_DIR = Path.cwd().parent
    SPLITS_DIR = REPO_DIR / "artifacts" / "splits"
    RUNS_BASE = REPO_DIR / "artifacts" / "runs"
    HARDWARE = "Local-CPU"

print("RUNS_BASE:", RUNS_BASE)

## 1. Imports e configuracao

In [ ]:
import json, time
from collections import Counter

import numpy as np
import pandas as pd

from economy_classifier.datasets import MULTICLASS_TOP7, OTHER_LABEL
from economy_classifier.ensemble import (
    majority_vote, weighted_vote, train_stacking_classifier, predict_stacking,
    compute_agreement_matrix, compute_fleiss_kappa,
)
from economy_classifier.evaluation import (
    compute_binary_metrics, compute_confusion_matrix,
    compute_cost_metrics, compute_multiclass_metrics, compute_roc_auc,
    summarize_cv_metrics,
)
from economy_classifier.project import build_result_card, persist_result_card

BASE_MODELS = [
    "tfidf_logreg",
    "tfidf_linearsvc",
    "tfidf_multinomialnb",
    "bert_bertimbau_base",
]
MULTI_LABELS = list(MULTICLASS_TOP7) + [OTHER_LABEL]

## 2. Loaders e helpers de alinhamento

In [ ]:
def card_path(model_id, task, regime):
    return RUNS_BASE / f"{model_id}_{task}_{regime}" / "result_card.json"

def predictions_path(model_id, task, regime):
    return RUNS_BASE / f"{model_id}_{task}_{regime}" / "predictions.csv"

def load_card(model_id, task, regime):
    p = card_path(model_id, task, regime)
    return json.loads(p.read_text()) if p.exists() else None

def load_preds(model_id, task, regime):
    p = predictions_path(model_id, task, regime)
    return pd.read_csv(p) if p.exists() else None

def align_by_index(preds_by_model, fold=None):
    base = None
    for model_id, df in preds_by_model.items():
        if fold is not None and "fold" in df.columns:
            df = df[df["fold"] == fold]
        df = df.set_index("index")
        cols = {"y_pred": f"y_pred_{model_id}"}
        if "y_score" in df.columns:
            cols["y_score"] = f"y_score_{model_id}"
        df = df.rename(columns=cols)
        keep = ["y_true", f"y_pred_{model_id}"]
        if f"y_score_{model_id}" in df.columns:
            keep.append(f"y_score_{model_id}")
        df = df[keep]
        base = df if base is None else base.join(df.drop(columns=["y_true"]), how="inner")
    return base.reset_index()

def cost_sum_from_cards(cards):
    return {
        "train_total": sum((c["cost"].get("train_seconds_mean") or 0.0) for c in cards.values()),
        "inf_total":   sum((c["cost"].get("inference_seconds_mean") or 0.0) for c in cards.values()),
        "size_total":  sum((c["cost"].get("model_size_mb") or 0.0) for c in cards.values()),
    }

## 3. BINARIO — voting (3 regimes)

Pesos do `weighted_vote` = F1 do regime correspondente do modelo base. Threshold fixo em 0.5.

In [ ]:
def f1_from_card(card):
    m = card["metrics"]
    return m.get("f1") or m.get("f1_mean") or 0.0

def run_binary_voting(regime):
    cards = {m: load_card(m, "binary", regime) for m in BASE_MODELS}
    preds = {m: load_preds(m, "binary", regime) for m in BASE_MODELS}
    if any(c is None or p is None for c, p in zip(cards.values(), preds.values())):
        missing = [m for m in BASE_MODELS if cards[m] is None or preds[m] is None]
        print(f"  SKIP {regime}: cards faltando para {missing}")
        return
    weights = {m: f1_from_card(cards[m]) for m in BASE_MODELS}
    cost_base = cost_sum_from_cards(cards)
    folds = sorted(preds[BASE_MODELS[0]]["fold"].unique()) if regime == "cv_5fold" else [None]

    for strategy in ["majority", "weighted"]:
        fold_metrics, all_out = [], []
        for fold in folds:
            aligned = align_by_index(preds, fold=fold)
            if strategy == "majority":
                pred_dict = {m: aligned[f"y_pred_{m}"] for m in BASE_MODELS}
                out = majority_vote(pred_dict, threshold=3)
                y_score = pd.Series(np.nan, index=aligned.index)
            else:
                score_dict = {m: aligned[f"y_score_{m}"] for m in BASE_MODELS if f"y_score_{m}" in aligned.columns}
                out = weighted_vote(score_dict, weights, threshold=0.5)
                y_score = out["y_score"]
            metrics = compute_binary_metrics(aligned["y_true"].values, out["y_pred"].values)
            if y_score.notna().all():
                metrics["auc_roc"] = round(compute_roc_auc(aligned["y_true"].values, y_score.values), 4)
            fold_metrics.append(metrics)
            df = pd.DataFrame({
                "index": aligned["index"], "y_true": aligned["y_true"],
                "y_pred": out["y_pred"], "method": f"ensemble_{strategy}",
            })
            if y_score.notna().all(): df["y_score"] = y_score
            if fold is not None: df["fold"] = fold
            all_out.append(df)
        agg = summarize_cv_metrics(fold_metrics) if regime == "cv_5fold" else fold_metrics[0]
        cost = compute_cost_metrics(
            train_seconds=cost_base["train_total"], inference_seconds=cost_base["inf_total"],
            n_inference_samples=len(all_out[0]),
            model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
        )
        run_dir = RUNS_BASE / f"ensemble_{strategy}_binary_{regime}"
        run_dir.mkdir(parents=True, exist_ok=True)
        pd.concat(all_out).to_csv(run_dir / "predictions.csv", index=False)
        card = build_result_card(
            model_id=f"ensemble_{strategy}", task="binary", regime=regime,
            metrics=agg, cost=cost,
            config={"base_models": BASE_MODELS,
                    "weights": weights if strategy == "weighted" else None,
                    "threshold": 0.5 if strategy == "weighted" else 3},
            n_train_samples=None, n_eval_samples=len(all_out[0]),
            predictions_path=str(run_dir / "predictions.csv"),
            notes=f"{strategy} vote across {len(BASE_MODELS)} base models",
        )
        persist_result_card(card, run_dir)
        primary = agg.get("f1") or agg.get("f1_mean")
        print(f"  {strategy:9s} {regime:14s} f1={primary}")

for regime in ["fixed_split", "cv_5fold", "test_set"]:
    print(f"\n=== BINARY {regime} ===")
    run_binary_voting(regime)

## 4. BINARIO — stacking (apenas `test_set`)

Meta treinado nos scores da `fixed_split` val, aplicado nas predicoes de `test_set` dos base models.

In [ ]:
cards_fs = {m: load_card(m, "binary", "fixed_split") for m in BASE_MODELS}
cards_ts = {m: load_card(m, "binary", "test_set") for m in BASE_MODELS}
preds_val = {m: load_preds(m, "binary", "fixed_split") for m in BASE_MODELS}
preds_test = {m: load_preds(m, "binary", "test_set") for m in BASE_MODELS}

if any(v is None for v in {**cards_fs, **cards_ts, **preds_val, **preds_test}.values()):
    print("SKIP stacking: result cards de fixed_split ou test_set faltando.")
else:
    aligned_val = align_by_index(preds_val)
    aligned_test = align_by_index(preds_test)
    val_scores = {m: aligned_val[f"y_score_{m}"] for m in BASE_MODELS}
    test_scores = {m: aligned_test[f"y_score_{m}"] for m in BASE_MODELS}
    t0 = time.perf_counter()
    meta = train_stacking_classifier(val_scores, aligned_val["y_true"], seed=42)
    meta_train_s = time.perf_counter() - t0
    out = predict_stacking(meta, test_scores)
    metrics = compute_binary_metrics(aligned_test["y_true"].values, out["y_pred"].values)
    metrics["auc_roc"] = round(compute_roc_auc(aligned_test["y_true"].values, out["y_score"].values), 4)
    cost_base = cost_sum_from_cards(cards_ts)
    cost = compute_cost_metrics(
        train_seconds=cost_base["train_total"] + meta_train_s,
        inference_seconds=cost_base["inf_total"],
        n_inference_samples=len(aligned_test),
        model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
    )
    run_dir = RUNS_BASE / "ensemble_stacking_binary_test_set"
    run_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "index": aligned_test["index"], "y_true": aligned_test["y_true"],
        "y_pred": out["y_pred"], "y_score": out["y_score"],
        "method": "ensemble_stacking",
    }).to_csv(run_dir / "predictions.csv", index=False)
    card = build_result_card(
        model_id="ensemble_stacking", task="binary", regime="test_set",
        metrics=metrics, cost=cost,
        config={"base_models": BASE_MODELS, "meta": "LogisticRegression",
                "meta_trained_on": "binary_fixed_split val scores"},
        n_train_samples=len(aligned_val), n_eval_samples=len(aligned_test),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="Stacking meta (LogReg) trained on val of fixed_split; applied to test_set scores",
    )
    persist_result_card(card, run_dir)
    print("BINARY stacking test_set:", json.dumps(metrics, indent=2))

## 5. MULTICLASSE — majority vote (3 regimes)

Empate vai para a classe predita pelo modelo individual com maior macro_F1.

In [ ]:
def macro_f1_from_card(card):
    m = card["metrics"]
    return m.get("macro_f1") or m.get("macro_f1_mean") or 0.0

def multiclass_majority(preds_wide, model_order_by_f1):
    cols = [f"y_pred_{m}" for m in model_order_by_f1]
    out = []
    for _, row in preds_wide[cols].iterrows():
        counts = Counter(row.tolist())
        max_v = max(counts.values())
        candidates = [c for c, v in counts.items() if v == max_v]
        if len(candidates) == 1:
            out.append(candidates[0])
        else:
            for c in cols:
                if row[c] in candidates:
                    out.append(row[c]); break
    return pd.Series(out, index=preds_wide.index)

def run_multi_voting(regime):
    cards = {m: load_card(m, "multiclass", regime) for m in BASE_MODELS}
    preds = {m: load_preds(m, "multiclass", regime) for m in BASE_MODELS}
    if any(c is None or p is None for c, p in zip(cards.values(), preds.values())):
        missing = [m for m in BASE_MODELS if cards[m] is None or preds[m] is None]
        print(f"  SKIP {regime}: cards faltando para {missing}")
        return
    order = sorted(BASE_MODELS, key=lambda m: macro_f1_from_card(cards[m]), reverse=True)
    cost_base = cost_sum_from_cards(cards)
    folds = sorted(preds[BASE_MODELS[0]]["fold"].unique()) if regime == "cv_5fold" else [None]
    fold_metrics, all_out = [], []
    for fold in folds:
        aligned = align_by_index(preds, fold=fold)
        y_pred = multiclass_majority(aligned, order)
        m = compute_multiclass_metrics(aligned["y_true"], y_pred, labels=MULTI_LABELS)
        fold_metrics.append(m)
        df = pd.DataFrame({
            "index": aligned["index"], "y_true": aligned["y_true"],
            "y_pred": y_pred, "method": "ensemble_majority_multi",
        })
        if fold is not None: df["fold"] = fold
        all_out.append(df)
    agg = summarize_cv_metrics(fold_metrics) if regime == "cv_5fold" else fold_metrics[0]
    cost = compute_cost_metrics(
        train_seconds=cost_base["train_total"], inference_seconds=cost_base["inf_total"],
        n_inference_samples=len(all_out[0]),
        model_size_mb=cost_base["size_total"], n_parameters=None, hardware=HARDWARE,
    )
    run_dir = RUNS_BASE / f"ensemble_majority_multiclass_{regime}"
    run_dir.mkdir(parents=True, exist_ok=True)
    pd.concat(all_out).to_csv(run_dir / "predictions.csv", index=False)
    if regime != "cv_5fold":
        cm = compute_confusion_matrix(all_out[0]["y_true"], all_out[0]["y_pred"], labels=MULTI_LABELS)
        cm.to_csv(run_dir / "confusion_matrix.csv")
    card = build_result_card(
        model_id="ensemble_majority", task="multiclass", regime=regime,
        metrics=agg, cost=cost,
        config={"base_models": BASE_MODELS, "tiebreak_order": order},
        n_train_samples=None, n_eval_samples=len(all_out[0]),
        predictions_path=str(run_dir / "predictions.csv"),
        notes="Mode across base models; tiebreak by descending individual macro_F1",
    )
    persist_result_card(card, run_dir)
    primary = agg.get("macro_f1") or agg.get("macro_f1_mean")
    print(f"  majority {regime:14s} macro_f1={primary}")

for regime in ["fixed_split", "cv_5fold", "test_set"]:
    print(f"\n=== MULTI {regime} ===")
    run_multi_voting(regime)

## 6. Concordancia entre base models (binary fixed_split)

In [ ]:
preds = {m: load_preds(m, "binary", "fixed_split") for m in BASE_MODELS}
if all(p is not None for p in preds.values()):
    aligned = align_by_index(preds)
    pred_dict = {m: aligned[f"y_pred_{m}"] for m in BASE_MODELS}
    cohen = compute_agreement_matrix(pred_dict)
    fleiss = compute_fleiss_kappa(pred_dict)
    print("Cohen's Kappa pareado:")
    print(cohen.round(3).to_string())
    print(f"\nFleiss Kappa global: {fleiss:.4f}")
else:
    print("SKIP agreement: predictions faltando.")

## 7. Sumario de cards do ensemble

In [ ]:
rows = []
for sub in sorted(RUNS_BASE.glob("ensemble_*")):
    cp = sub / "result_card.json"
    if not cp.exists(): continue
    c = json.loads(cp.read_text())
    m = c["metrics"]
    primary = m.get("f1") or m.get("f1_mean") or m.get("macro_f1") or m.get("macro_f1_mean")
    rows.append({
        "strategy": c["model_id"], "task": c["task"], "regime": c["regime"],
        "primary": primary,
        "train_total_s": c["cost"].get("train_seconds_mean"),
        "size_total_mb": c["cost"].get("model_size_mb"),
    })
pd.DataFrame(rows).sort_values(["task", "regime", "strategy"]).reset_index(drop=True)